In [ ]:
# Import libraries
import os
import zipfile
import sqlite3
import requests
import pandas as pd

In [ ]:
# Step 1: Download and save the ZIP file from Zenodo
zenodo_url = "https://zenodo.org/records/11648429/files/processed%20data.zip?download=1"
zip_filename = "processed_data.zip"

print("Downloading dataset...")
response = requests.get(zenodo_url)
with open(zip_filename, "wb") as f:
    f.write(response.content)

In [ ]:
# Step 2: Extract the ZIP file
extract_dir = "extracted_data"
with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

In [ ]:
# Step 3: Load the CSV into a SQLite database
csv_path = os.path.join(extract_dir, "record_txt_unique.csv")
db_path = "data.db"

conn = sqlite3.connect(db_path)
chunk_size = 10_000

print("Creating SQLite database...")
for chunk in pd.read_csv(csv_path, chunksize=chunk_size):
    chunk.to_sql("TBLPapers", conn, if_exists="append", index=False)

In [ ]:
# Step 4: Add unique paper ID
cursor = conn.cursor()
cursor.execute("""
    CREATE TABLE TBLPapers_New AS
    SELECT 
        ROW_NUMBER() OVER (ORDER BY (SELECT NULL)) AS IDPaper,
        * 
    FROM TBLPapers;
""")

cursor.execute("DROP TABLE TBLPapers")
cursor.execute("ALTER TABLE TBLPapers_New RENAME TO TBLPapers")

conn.commit()
conn.close()

In [ ]:
# Step 5: Remove downloaded files safely
try:
    if os.path.exists(zip_filename):
        os.remove(zip_filename)
    if os.path.exists(csv_path):
        os.remove(csv_path)
    if os.path.isdir(extract_dir):
        os.rmdir(extract_dir)
    print("Cleanup completed. Database created: data.db")
except Exception as e:
    print(f"Cleanup warning: {e}")
